# Chapter 2 companion: block diagrams and Mason's gain formula.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.

Reproduces:
  * the robot-arm position loop: J_eq, B_eq, K, a, the closed loop
    25/(s^2 + 6s + 25) and the effect of keeping the inductance L = 0.01 H;
  * Mason's gain formula, implemented for any signal flow graph whose branch
    gains are numbers (or functions of s evaluated at one point).  It lists the
    forward paths and loops by depth-first search, builds Delta and the
    cofactors, and is checked against a direct solution of the node equations;
  * Example 2 (two forward paths, two loops that do not touch): 5/8;
  * Example 3 (three interlaced loops) against the block-diagram reduction;
  * the rate-feedback loop: K/(s^2 + 2s + K).

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
# In the browser (JupyterLite), hide a harmless Matplotlib warning from its in-browser backend.
import sys, warnings
if "pyodide" in sys.modules:
    warnings.filterwarnings("ignore", message=r"The \w+ parameter as float was deprecated")

In [ ]:
from itertools import combinations
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

In [ ]:
# ---------------------------------------------------------------- robot arm
Jm = Bm = 0.01

In [ ]:
Jarm = Barm = 4.0
n, R, Kt, Km = 0.1, 1.0, 0.5, 0.5
J = Jm + n**2 * Jarm
B = Bm + n**2 * Barm
K = n * Kt / (R * J)
a = (R * B + Kt * Km) / (R * J)
print("Robot arm, reflected to the motor shaft")
print(f"  J_eq = {J:.3f}, B_eq = {B:.3f}, K = {K:.3f}, a = {a:.3f}  ->  G(s) = {K:g}/(s(s+{a:g}))")
Gc = 25.0
den = [1, a, K * Gc]
poles = np.roots(den)
wn = np.sqrt(den[2])
print(f"  G_c = 25: closed loop 25/(s^2 + {a:g}s + 25), poles {np.round(poles, 4)},"
      f" wn = {wn:g} rad/s, zeta = {a / (2 * wn):g}")
L = 0.01
motor = np.polyadd(np.polymul([L, R], [J, B]), [Kt * Km])      # (R + Ls)(Js + B) + Kt Km
print(f"  with L = 0.01 H: (R+Ls)(Js+B) + KtKm = {motor[0]:g} (s^2 + {motor[1] / motor[0]:g}s"
      f" + {motor[2] / motor[0]:g}), poles of G: 0, {np.round(np.roots(motor), 2)}")
print(f"  equal real closed-loop poles: G_c = a^2/4 = {a**2 / 4:g}")

In [ ]:
# ---------------------------------------------------------------- Mason's formula
def paths_and_loops(branches, src, snk):
    """branches: dict {(from, to): gain}.  Returns forward paths and loops as
    lists of (node list, gain)."""
    out = {}
    for (u, v), g in branches.items():
        out.setdefault(u, []).append((v, g))
    nodes = sorted({u for u, _ in branches} | {v for _, v in branches})
    paths = []

    def dfs(node, visited, gain):
        if node == snk:
            paths.append((visited[:], gain))
            return
        for v, g in out.get(node, []):
            if v not in visited:
                dfs(v, visited + [v], gain * g)
    dfs(src, [src], 1.0)

    loops = []
    order = {v: i for i, v in enumerate(nodes)}

    def dfs_loop(start, node, visited, gain):
        for v, g in out.get(node, []):
            if v == start:
                loops.append((visited[:], gain * g))
            elif v not in visited and order[v] > order[start]:
                dfs_loop(start, v, visited + [v], gain * g)
    for s0 in nodes:                         # each loop is found once, from its lowest node
        dfs_loop(s0, s0, [s0], 1.0)
    return paths, loops

In [ ]:
def delta(loops):
    """1 - sum L_i + sum of non-touching pairs - ... ."""
    total = 1.0
    for r in range(1, len(loops) + 1):
        for combo in combinations(loops, r):
            sets = [set(nl) for nl, _ in combo]
            if all(not (sets[i] & sets[j]) for i in range(r) for j in range(i + 1, r)):
                total += (-1) ** r * np.prod([g for _, g in combo])
    return total

In [ ]:
def mason(branches, src, snk):
    paths, loops = paths_and_loops(branches, src, snk)
    D = delta(loops)
    num = sum(g * delta([lp for lp in loops if not set(lp[0]) & set(nodes)]) for nodes, g in paths)
    return num / D, paths, loops

In [ ]:
def node_equations(branches, src, snk):
    """Solve x = A x + b x_src directly (each node is the sum of what enters it)."""
    nodes = sorted({u for u, _ in branches} | {v for _, v in branches})
    idx = {v: i for i, v in enumerate(nodes)}
    A = np.zeros((len(nodes), len(nodes)), complex)
    for (u, v), g in branches.items():
        A[idx[v], idx[u]] += g
    M = np.eye(len(nodes)) - A
    M[idx[src], :] = 0
    M[idx[src], idx[src]] = 1
    rhs = np.zeros(len(nodes), complex)
    rhs[idx[src]] = 1
    return np.linalg.solve(M, rhs)[idx[snk]]

In [ ]:
# Example 2: two forward paths, two loops that do not touch
def example2(G1, G2, G3, G4, G5, G6, H1, H2):
    return {("R", "x1"): G1, ("x1", "x2"): G2, ("x2", "x1"): -H1, ("x2", "x3"): G3,
            ("R", "x3"): G6, ("x3", "x4"): G4, ("x4", "x3"): -H2, ("x4", "C"): G5}

In [ ]:
br = example2(1, 1, 1, 1, 1, 2, 1, 3)
T, paths, loops = mason(br, "R", "C")
print("\nExample 2 with G1..G5 = 1, G6 = 2, H1 = 1, H2 = 3")
print("  forward paths:", [("-".join(p), g) for p, g in paths])
print("  loops:        ", [("-".join(p), g) for p, g in loops])
print(f"  Mason: T_CR = {T:.4f} (5/8 = {5 / 8}),  node equations: {node_equations(br, 'R', 'C').real:.4f}")

In [ ]:
# Example 3: three interlaced loops; compare with the reduced formula at s = 0.7 + 1.3j
def blocks(s):
    G1, G2, G3, G4 = 2 / (s + 1), 3 / (s + 2), 1 / s, 4 / (s + 3)
    H1, H2, H3 = 0.5, s / (s + 5), 1.0
    return G1, G2, G3, G4, H1, H2, H3

In [ ]:
for s in (0.7 + 1.3j, 2.0 + 0j):
    G1, G2, G3, G4, H1, H2, H3 = blocks(s)
    br3 = {("R", "E1"): 1, ("E1", "E2"): G1, ("E2", "E3"): G2, ("E3", "y"): G3, ("y", "C"): G4,
           ("C", "E3"): H1, ("y", "E2"): -H2, ("C", "E1"): -H3}
    Tm, _, loops3 = mason(br3, "R", "C")
    Tred = G1 * G2 * G3 * G4 / (1 - G3 * G4 * H1 + G2 * G3 * H2 + G1 * G2 * G3 * G4 * H3)
    print(f"\nExample 3 at s = {s}: {len(loops3)} loops; Mason {Tm:.6f}, reduction {Tred:.6f},"
          f" node equations {node_equations(br3, 'R', 'C'):.6f}")

In [ ]:
# exit ticket: tachometer (rate) feedback inside a unity position loop, plant 1/(s(s+1))
#   R -> e -> [K] -> x -> [1/(s+1)] -> v (speed) -> [1/s] -> C,  v -> x with -1,  C -> e with -1
Kg = 4.0

In [ ]:
for s in (0.5 + 0.5j, 3.0 + 0j):
    brt = {("R", "e"): 1, ("e", "x"): Kg, ("x", "v"): 1 / (s + 1), ("v", "C"): 1 / s,
           ("v", "x"): -1, ("C", "e"): -1}
    Tm, _, _ = mason(brt, "R", "C")
    print(f"Rate feedback, K = {Kg:g}, s = {s}: Mason {Tm:.6f},  K/(s^2+2s+K) = {Kg / (s * s + 2 * s + Kg):.6f}")

In [ ]:
# ---------------------------------------------------------------- figure: arm step responses
t = np.linspace(0, 2.5, 1000)

In [ ]:
_, y2 = signal.step(([25], den), T=t)
G3 = (np.array([n * Kt]), np.polymul(motor, [1, 0]))
cl3 = (G3[0] * Gc, np.polyadd(G3[1], G3[0] * Gc))
_, y3 = signal.step(cl3, T=t)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(t, y2, label="L neglected: 25/(s^2+6s+25)")
ax.plot(t, y3, "--", label="L = 0.01 H (third order)")
ax.set(xlabel="t (s)", ylabel="arm angle / command", title="Robot-arm position loop, G_c = 25")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ch02_arm_step.png", dpi=150)
print(f"\nPeak overshoot: {100 * (y2.max() - 1):.1f}% (L neglected), {100 * (y3.max() - 1):.1f}% (L kept)")
print("Saved figures/ch02_arm_step.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**The robot-arm position loop.** The loop is $\Theta_L/\Theta_c = GG_c/(1+GG_c)$, the
one-loop case of Mason's formula. With the inductance neglected, $G(s) = 1/(s(s+6))$ and
$G_c = k$ give $k/(s^2 + 6s + k)$. Move $k$: at $k = 9$ the two poles meet at $-3$, at
$k = 25$ they are $-3 \pm j4$. Then raise the inductance $L$ and push $k$ up: the
third-order loop, which keeps $L$, eventually goes unstable, while the second-order model
never does.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

# reflected to the motor shaft (SI units)
J = B = 0.05
R, KT, KM, N = 1.0, 0.5, 0.5, 0.1


def _overshoot(y):
    """Percent overshoot over the final value 1 (both loops have DC gain 1)."""
    return max(0.0, 100 * (y.max() - 1))


def explore(k=25.0, L_mH=10.0):
    L = L_mH / 1000
    # L neglected: k/(s^2 + 6s + k)
    den2 = np.array([R * J, R * B + KT * KM, N * KT * k]) / (R * J)
    num2 = [N * KT * k / (R * J)]
    # L kept: n Kt k / (s[(R + Ls)(Js + B) + Kt Km] + n Kt k)
    motor = np.polyadd(np.polymul([L, R], [J, B]), [KT * KM])
    den3 = np.polyadd(np.polymul(motor, [1, 0]), [N * KT * k])
    den3 = np.trim_zeros(den3, "f")
    num3 = [N * KT * k]
    p2, p3 = np.roots(den2), np.roots(den3)
    stable3 = np.all(p3.real < -1e-12)

    slow = np.min(np.abs(p2.real))
    t_end = float(np.clip(8 / slow, 2.0, 30.0))
    t = np.linspace(0, t_end, 1500)
    _, y2 = signal.step((num2, den2), T=t)
    _, y3 = signal.step((num3, den3), T=t)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0),
                                   gridspec_kw={"width_ratios": [1.8, 1]})
    ax1.plot(t, y2, color=ORANGE, lw=2, label="L neglected (second order)")
    ax1.plot(t, y3, "--", color=BLUE, lw=1.6, label=f"L = {L_mH:g} mH kept (third order)")
    ax1.axhline(1, color="k", lw=0.6)
    hi = max(1.8, y2.max() + 0.1)
    ax1.set_ylim(-0.3 if stable3 else -1.5, hi if stable3 else 3.0)
    ax1.set_xlim(0, t_end)
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel(r"arm angle $\theta_L$ / command")
    ax1.set_title(f"Unit step command, $G_c = k = {k:g}$", fontsize=11)
    ax1.legend(fontsize=10, loc="upper right")

    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    ax2.plot(p2.real, p2.imag, "x", color=ORANGE, ms=11, mew=2.5, label="L neglected")
    ax2.plot(p3.real, p3.imag, "o", mfc="none", color=BLUE, ms=10, mew=1.8, label="L kept")
    ax2.set_xlim(-12, 3)
    ymax = max(6.0, 1.2 * np.max(np.abs(np.r_[p2.imag, p3.imag])))
    ax2.set_ylim(-ymax, ymax)
    ax2.set_xlabel("Re s (1/s)")
    ax2.set_ylabel("Im s (rad/s)")
    ax2.set_title("closed-loop poles (fast pole may be off scale)", fontsize=10)
    ax2.legend(fontsize=10, loc="upper left")
    fig.tight_layout()

    fmt = lambda p: ", ".join(f"{z.real:.2f}{z.imag:+.2f}j" for z in p)
    wn = np.sqrt(k)
    print(f"L neglected: k/(s^2 + 6s + k), poles {fmt(p2)}; wn = {wn:.2f} rad/s, "
          f"zeta = {3 / wn:.2f}, overshoot {_overshoot(y2):.1f}%")
    if stable3:
        print(f"L = {L_mH:g} mH kept: poles {fmt(p3)}, overshoot {_overshoot(y3):.1f}%")
    else:
        print(f"L = {L_mH:g} mH kept: poles {fmt(p3)}: UNSTABLE")
    if L > 0:
        # Routh on L J s^3 + (R J + L B) s^2 + (R B + Kt Km) s + n Kt k
        kmax = (R * J + L * B) * (R * B + KT * KM) / (L * J * N * KT)
        print(f"third-order loop is stable for 0 < k < {kmax:.0f}; the second-order model for every k > 0")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             k=FloatSlider(value=25.0, min=1.0, max=200.0, step=1.0, description="k = G_c",
                           continuous_update=False),
             L_mH=FloatSlider(value=10.0, min=0.0, max=50.0, step=1.0, description="L (mH)",
                              continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 2.** These cells go with the Hands-on exercises at the end of
Chapter 2. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
# ---------------------------------------------------------------- H1
def h1_sensor_offset(K=3.8):
    """H1: nozzle heater G = 0.5/(s+0.1), gain K; fan d = -4 W at t = 1 s, offset v = 3 C at t = 4 s."""
    t = np.linspace(0, 8, 4001)
    d = np.where(t >= 1, -4.0, 0.0)
    n = np.where(t >= 4, 3.0, 0.0)
    # y' = -0.1 y + 0.5 (K (r - y - n) + d), with r = 0 (deviation from the setpoint)
    sys = signal.StateSpace([[-0.1 - 0.5 * K]], [[0.5, -0.5 * K]], [[1.0]], [[0.0, 0.0]])
    _, y, _ = signal.lsim(sys, np.column_stack([d, n]), t)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, y, color=ORANGE, lw=1.8, label="true nozzle temperature y")
    ax.plot(t, y + n, "--", color=BLUE, lw=1.4, label="measured y + v")
    ax.axhline(0, color="k", lw=0.6)
    ax.axvline(1, color=GRAY, lw=0.8, ls=":")
    ax.axvline(4, color=GRAY, lw=0.8, ls=":")
    ax.text(1.05, 2.8, "fan on", color=GRAY, fontsize=9)
    ax.text(4.05, 2.8, "offset appears", color=GRAY, fontsize=9)
    ax.set_ylim(-4.5, 3.5)
    ax.set_xlabel("t (s)")
    ax.set_ylabel("deviation (C)")
    pole = -(0.1 + 0.5 * K)
    ax.set_title(f"K = {K:g} W/C: pole {pole:.2f}, fan step {y[t < 4][-1]:.3f} C, "
                 f"offset step {y[-1] - y[t < 4][-1]:.3f} C", fontsize=11)
    ax.legend(loc="lower left", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_node_check(H1=1.0, F=1.0):
    """H2: the graph with two loops meeting at x2 (G1 = G2 = 1); node equations vs both formulas."""
    G1 = G2 = 1.0
    H2s = np.linspace(0, 4, 41)
    direct = []
    for H2 in H2s:
        # unknowns x1, x2, x3 with R = 1
        A = np.array([[1, H1, 0], [-G1, 1, H2], [0, -G2, 1]], dtype=float)
        x = np.linalg.solve(A, [1, 0, F])
        direct.append(x[2])
    good = (G1 * G2 + F * (1 + G1 * H1)) / (1 + G1 * H1 + G2 * H2s)
    bad = (G1 * G2 + F) / ((1 + G1 * H1) * (1 + G2 * H2s))
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(H2s, direct, "o", color=ORANGE, ms=6, label="node equations solved directly")
    ax.plot(H2s, good, color=BLUE, lw=1.6, label="corrected Mason formula")
    ax.plot(H2s, bad, "--", color=TEAL, lw=1.4, label="classmate's formula")
    ax.set_xlabel("$H_2$")
    ax.set_ylabel("$C/R$")
    ax.set_title(f"$G_1 = G_2 = 1$, $H_1$ = {H1:g}, F = {F:g}", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_drone(Kp=2.0, Kv=8.0):
    """H3: altitude loop h'' = a + D, v_c = Kp (h_c - h), a = Kv (v_c - h'); 1 m step, then D = -0.6."""
    den = [1, Kv, Kp * Kv]
    t = np.linspace(0, 8, 2001)
    _, h_ref = signal.step(signal.TransferFunction([Kp * Kv], den), T=t)
    t2 = t[t <= 4]
    _, h_d = signal.step(signal.TransferFunction([-0.6], den), T=t2)
    h = h_ref.copy()
    h[t >= 4] += h_d[: np.count_nonzero(t >= 4)]
    poles = np.roots(den)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(t, h, color=ORANGE, lw=1.6)
    ax1.axhline(1, color="k", lw=0.6)
    ax1.axvline(4, color=GRAY, lw=0.8, ls=":")
    ax1.text(4.05, 0.2, "downdraft -0.6 m/s$^2$", color=GRAY, fontsize=9)
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel("altitude h (m)")
    over = max(0.0, (h_ref.max() - 1) * 100)
    ax1.set_title(f"Kp = {Kp:g}, Kv = {Kv:g}: overshoot {over:.2f}%, "
                  f"offset {100 * (1 - h[-1]):.2f} cm", fontsize=11)
    ax2.axhline(0, color=GRAY, lw=0.8)
    ax2.axvline(0, color=GRAY, lw=0.8)
    ax2.plot(poles.real, poles.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax2.set_xlim(-25, 2)
    ax2.set_ylim(-14, 14)
    kind = "real" if Kv * Kv - 4 * Kp * Kv >= -1e-9 else "complex"   # discriminant
    ax2.set_title(f"poles ({kind})", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_gearbox(ratio=50, k=6.0):
    """H4: pan head through a reducer n = 1/ratio; P control k; checks slew, overshoot, settling."""
    Jm, JL, R, Kt, Km, wmax = 0.001, 0.5, 0.6, 0.06, 0.06, 250.0
    n = 1.0 / ratio
    J = Jm + n ** 2 * JL
    a = Kt * Km / (R * J)
    K = n * Kt / (R * J)
    den = [1, a, k * K]
    poles = np.roots(den)
    wn = np.sqrt(k * K)
    zeta = a / (2 * wn)
    t = np.linspace(0, 6, 3001)
    _, th = signal.step(signal.TransferFunction([k * K], den), T=t)
    outside = np.nonzero(np.abs(th - 1) > 0.02)[0]
    ts = t[outside[-1]] if len(outside) else 0.0
    slew = wmax * n
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, th, color=ORANGE, lw=1.8, label="head angle, 1 rad step")
    ax.axhspan(0.98, 1.02, color=BLUE, alpha=0.15, label="2% band")
    ax.axhline(1.05, color=TEAL, ls="--", lw=1, label="5% overshoot")
    ax.axvline(2, color=GRAY, ls=":", lw=1, label="t = 2 s")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("$\\theta_L$ (rad)")
    ok = lambda c: "pass" if c else "FAIL"
    ax.set_title(f"1:{ratio}: a = {a:.2f}, poles {poles[0]:.2f}, {poles[1]:.2f}; "
                 f"slew {slew:.2f} rad/s ({ok(slew >= 4)}), zeta {zeta:.2f} ({ok(zeta >= 0.7)}), "
                 f"slowest pole {max(poles.real):.2f} ({ok(max(poles.real) <= -2 + 1e-9)}), "
                 f"settles in {ts:.2f} s", fontsize=9)
    ax.legend(loc="lower right", fontsize=9)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(h1_sensor_offset, K=FloatSlider(value=3.8, min=0.2, max=60, step=0.2,
                                             description="K (W/C)", continuous_update=False))
    interact(h2_node_check,
             H1=FloatSlider(value=1.0, min=0, max=3, step=0.1, description="H1",
                            continuous_update=False),
             F=FloatSlider(value=1.0, min=-2, max=3, step=0.1, description="F",
                           continuous_update=False))
    interact(h3_drone,
             Kp=FloatSlider(value=1.0, min=0.1, max=8, step=0.1, description="Kp (1/s)",
                            continuous_update=False),
             Kv=FloatSlider(value=8.0, min=1, max=24, step=0.5, description="Kv (1/s)",
                            continuous_update=False))
    interact(h4_gearbox,
             ratio=IntSlider(value=20, min=5, max=100, description="N_L/N_m",
                             continuous_update=False),
             k=FloatSlider(value=6.0, min=0.5, max=20, step=0.1, description="k (V/rad)",
                           continuous_update=False))
except ImportError:
    h1_sensor_offset()
    h2_node_check()
    h3_drone()
    h4_gearbox()